# 04 WoE scorecard (champion)

Purpose: build the logistic scorecard on the 10 confirmed features.
Steps: propose cut-offs from training quantiles, compute WoE, check monotonicity,
fit logistic regression, check coefficient signs, scale to points, confirm the points
reproduce the model PD, and produce reason codes.

Cut-offs are proposals. Each one needs your confirmation before it is written into
`src/config.py`. Settings in use: base score 600, base odds 50:1, PDO 20.

In [ ]:
import sys, importlib
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import src.scorecard as sc
importlib.reload(sc)

from src import config
from src.splits import assign_sample
from src.feature_selection import add_derived_features
from src.metrics import gini, ks_statistic, brier_score
from src.feature_selection import vif

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

raw = assign_sample(pd.read_parquet((lambda p: p if p.exists() else config.PROCESSED_PATH)(getattr(config, "SLIM_PATH", config.PROCESSED_PATH.with_name("loans_slim.parquet")))))
df = add_derived_features(raw[raw["sample"] != "excluded"].copy())
train = df[df["sample"] == "train"].reset_index(drop=True)
valid = df[df["sample"] == "validation"].reset_index(drop=True)
oot = df[df["sample"] == "oot"].reset_index(drop=True)
print(f"train {len(train):,} | validation {len(valid):,} | oot {len(oot):,}")
print("module loaded from:", sc.__file__)

## 1. Proposed cut-offs (confirm before use)

Numeric features get cut-offs at the training quartiles and tertiles, rounded.
Categorical features are used as they are. Adjust any cut-off here for business
sense, then rerun. Monotonicity and IV are checked in the next sections.

In [ ]:
assert "train" in globals() and "oot" in globals(), "Run the setup cell (the first code cell, which loads the data) before this cell."
from src import config
import numpy as np
import pandas as pd
NUMERIC = [f for f in config.SCORECARD_FEATURES if f not in config.CATEGORICAL_SCORECARD]

def propose_cuts(s, n_groups=5, decimals=1):
    q = np.quantile(s.dropna(), np.linspace(0, 1, n_groups + 1)[1:-1])
    return sorted(set(np.round(q, decimals)))

proposal = {f: propose_cuts(train[f], n_groups=5, decimals=0 if f in ("fico", "annual_inc", "credit_history_months") else 1)
            for f in NUMERIC}
for f, cuts in proposal.items():
    print(f"{f:28s} {cuts}")

SPEC = {f: {"type": "numeric", "cuts": proposal[f]} for f in NUMERIC}
KEEP = {f: sorted(train[f].fillna("missing").astype(str).value_counts(normalize=True).loc[lambda s: s >= config.RARE_CATEGORY_SHARE].index) for f in config.CATEGORICAL_SCORECARD}
for f in config.RARE_MERGE_EXEMPT:
    KEEP[f] = sorted(train[f].fillna("missing").astype(str).unique())
print({f: len(v) for f, v in KEEP.items()}, "levels kept")
SPEC.update({f: {"type": "categorical", "keep": KEEP[f]} for f in config.CATEGORICAL_SCORECARD})

## 2. Bins, WoE and bad rate by bin (training sample)

WoE is positive where a bin has relatively more goods, so it means lower risk.
A bin with WoE far from zero and a sensible bad rate is doing real work.

In [ ]:
assert "train" in globals() and "oot" in globals(), "Run the setup cell (the first code cell, which loads the data) before this cell."
train_bins, orders = sc.assign_bins(train, SPEC)
valid_bins, _ = sc.assign_bins(valid, SPEC)
oot_bins, _ = sc.assign_bins(oot, SPEC)

maps = sc.woe_maps(train_bins, train["target"])
for f in SPEC:
    tab = (pd.DataFrame({"bin": train_bins[f].values, "y": train["target"].values})
           .groupby("bin")["y"].agg(loans="size", bad_rate="mean").reset_index())
    tab["woe"] = tab["bin"].map(maps[f]).round(3)
    print(f"\n== {f}")
    display(tab.round(4))

## 3. Monotonicity check

Expected direction comes from `config.MONOTONIC_SIGN`. A violation means the bins
zig-zag against business logic. The fix is to merge the bins that break the order,
and the result is recorded in `docs/feature_decisions.md`.

In [ ]:
assert "train" in globals() and "oot" in globals(), "Run the setup cell (the first code cell, which loads the data) before this cell."
mono = sc.monotonic_check(train_bins, orders, train["target"], SPEC)
display(mono)
print("All numeric features monotonic:", bool(mono["monotonic"].all()))

## 4. Fit the logistic regression on WoE

The regularisation strength C is chosen on the validation sample by log loss.
Every coefficient should be negative, because WoE already points the right way.
A positive coefficient means the feature is working against its own bins, and
needs a look before we go on.

In [ ]:
assert "train" in globals() and "oot" in globals(), "Run the setup cell (the first code cell, which loads the data) before this cell."
fit = sc.fit_scorecard(train_bins, train["target"], valid_bins, valid["target"])
print(f"Chosen C: {fit['C']}   validation log loss: {fit['valid_logloss']:.5f}")
coef = pd.DataFrame({"coefficient": pd.Series(fit["coef"]), "sign_ok": pd.Series(fit["coef"]) < 0})
display(coef.round(4))
print("intercept:", round(fit["intercept"], 4))
print("All coefficients negative:", bool(coef["sign_ok"].all()))

## 5. Points scaling

Points per bin = -factor x coefficient x WoE, where factor = PDO / ln(2).
Base points are the score with every feature at zero WoE.

In [ ]:
assert "train" in globals() and "oot" in globals(), "Run the setup cell (the first code cell, which loads the data) before this cell."
pts, base = sc.points_table(fit)
print(f"factor = {sc.FACTOR:.4f}   offset = {sc.OFFSET:.4f}   base points = {base:.2f}")
display(pts.round(3).sort_values(["feature", "points"]).reset_index(drop=True))

## 6. Check: points reproduce the model PD

This must hold. If the points-based PD and the model PD differ, the scaling or the
sign convention is wrong, and nothing downstream should be used.

In [ ]:
assert "train" in globals() and "oot" in globals(), "Run the setup cell (the first code cell, which loads the data) before this cell."
pd_model = sc.pd_from_model(fit, oot_bins)
pd_points = sc.pd_from_score(sc.score(fit, oot_bins))
max_gap = float(np.max(np.abs(pd_model - pd_points)))
print(f"max |PD_model - PD_points| on out-of-time: {max_gap:.2e}")
assert max_gap < 1e-9, "points do not reproduce the model PD"
print("Check passed.")

## 7. Preliminary discrimination (full validation comes in Stage 6)

These are first readings to confirm the scorecard works. They use the same
validation and out-of-time samples as the Stage 6 report, which will also cover
calibration, decile tables and the cut-off analysis.

In [ ]:
assert "train" in globals() and "oot" in globals(), "Run the setup cell (the first code cell, which loads the data) before this cell."
s_valid = sc.score(fit, valid_bins)
s_oot = sc.score(fit, oot_bins)
pd_valid = sc.pd_from_model(fit, valid_bins)
pd_oot = sc.pd_from_model(fit, oot_bins)

summary = pd.DataFrame({
    "sample": ["validation", "out-of-time"],
    "gini": [gini(valid["target"], -s_valid), gini(oot["target"], -s_oot)],
    "ks": [ks_statistic(valid["target"], pd_valid), ks_statistic(oot["target"], pd_oot)],
    "bad_rate": [valid["target"].mean(), oot["target"].mean()],
    "mean_pd": [pd_valid.mean(), pd_oot.mean()],
    "brier": [brier_score(valid["target"], pd_valid), brier_score(oot["target"], pd_oot)],
})
display(summary.round(4))

Read the mean PD against the bad rate: the gap on out-of-time is the calibration
drift found in Stage 2. It is reported in full in Stage 6, not fixed here.

## 8. VIF on the final 10 features

In [ ]:
assert "train" in globals() and "oot" in globals(), "Run the setup cell (the first code cell, which loads the data) before this cell."
numeric_final = [f for f in NUMERIC]
display(vif(train, numeric_final).round(2).to_frame("vif"))

After `loan_to_income` was dropped, every VIF should fall below 10. Any feature above
10 goes back to the decision record before the scorecard is accepted.

## 9. Reason codes for sample applicants (out-of-time)

In [ ]:
assert "train" in globals() and "oot" in globals(), "Run the setup cell (the first code cell, which loads the data) before this cell."
sample_rows = oot_bins.sample(3, random_state=7)
reasons = sc.reason_codes(fit, sample_rows, top_n=4)
out = pd.DataFrame({
    "score": sc.score(fit, sample_rows).round(1),
    "pd": sc.pd_from_model(fit, sample_rows).round(4),
    "reasons": reasons.values,
}, index=sample_rows.index)
display(out)

Reason codes list the features where the applicant scores furthest below the best
bin for that feature. They are the basis of adverse action notices.